# Run Options

In [ ]:
RUN_FRAGMENTATION = True
MOLECULE = 'H2S'
N_MODES = 3 # Number of Modes 
TRUNCATION = 4
MAX_NUM_FRAGMENTS = 6 # for fragmentation

# VQD Parameters:
num_states = 5 # Number of states to compute.
ndepth = 20 # Number of layers in the qumode circuit ansatz.
niter = 1 # Maximum number of iterations for the optimization.
threshold = 1e-3 # Convergence threshold for the optimization.
nfocks = [16, 16] # Designates the number of modes and number of fock states for each mode. For example, [16] means 1 mode with 16 fock states, and [16, 16] means 2 modes with 16 fock states each.

In [2]:
from pathlib import Path
fragment_path = Path.cwd().parent / f'data/{MOLECULE}_Params_V3.pkl' # if RUN_FRAGMENTATION, save to this location, else read fragments from this location 
hquad_path = Path.cwd().parent / f'data/{MOLECULE}_Hquad_V3.pkl' # save quadratic Hamiltonian to this location

# Check for Necessary Files

In [3]:
if not RUN_FRAGMENTATION:
    if not fragment_path.exists():
        raise FileNotFoundError(f"Fragment file not found at {fragment_path}.\n"
                                "Please run the fragmentation script first by setting RUN_FRAGMENTATION to True.\n"
                                "Alternativley, download the files from https://github.com/sdonglab/QumVQD.git")

In [ ]:
if MOLECULE == 'H2S':
    p = Path("..") / "data" / "H2S.hdf5"
elif MOLECULE == 'CO2':
    p = Path("..") / "data" / "CO2.hdf5"

if not p.exists():
    raise FileNotFoundError(f"Vibrational Hamiltonian file not found at  {p}.\n"
                            "Please download the file from https://github.com/sdonglab/QumVQD.git and put it in the data directory.")

# Prerequisite

In [5]:
from silence_tensorflow import silence_tensorflow
silence_tensorflow()

I0000 00:00:1779412056.384266  379330 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779412056.391160  379330 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779412061.895802  379330 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779412061.899702  379330 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [6]:
import tensorflow as tf

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

TensorFlow version: 2.21.0
GPU available: []


# Vibrational Hamiltonian

## prereqs

In [7]:
import numpy as np
import time

from third_party.vibrational_dynamics.util_hamil import operator_quad, hamil
from third_party.vibrational_dynamics.util_gfro import (
    norm_m,
    obtain_fragment,
    fragments,
    subtract,
    non_rotated_fragments,
    recreate_matrices,
    vectors,
    super_matrix_exp,
    extract_sub_matrices,
    tranformed_operators,
    compute_fragment,
    initial,
    flatten_matrices,
)
from third_party.vibrational_dynamics.util_qudratic import b_transform, recreate_quadH

from scipy.optimize import minimize



# Hamiltonian in the form of OpenFermion bosonic operator can be created with two functions
# "operator_quad" function creates hamiltonian generated with Gaussian
# "hamil" function creates hamiltonian provided by Xanadu

name = MOLECULE

# H = operator_quad(name) # Hamiltonian in OpenFermion bosonic operator form
# Options are "CO","CO2","NH3",
# model_hamiltonians options "D2","D3"

H = hamil(name)  # Hamiltonian in OpenFermion bosonic operator form
# Options are "H2S","CO2","CH4","H2S_3c","CO2_3c"
# "H2S_3c","CO2_3c" are with 3 mode coupling


n = N_MODES  # Number of modes

params = list()
norms = list()


# calculating the cost function(sum of square of the coeffiecients of H-H_f operator)


def cost(X):

    P, Q, G, L = recreate_matrices(X, n)  # Calculate P,Q,G,L from paramter vector X

    expA = super_matrix_exp(P, Q)  # Calculate exp([[P,Q],[Q,P]])

    U, V = extract_sub_matrices(
        expA, n
    )  # Extract U and V from exp([[P,Q],[Q,P]]) = [[U,-V],[-V,U]]

    C_b, C_bd = tranformed_operators(U, V, G)  # Calculate transformed operators

    H_f = compute_fragment(
        L, C_b, C_bd, n
    )  # Calculate quartic fragments from calculated L, C_b, C_bd

    H_vector, H_f_vector, sorted_ops = vectors(
        H, H_f
    )  # extract coeffiecients and corresponding operators from H and H_f

    # Finding indices corresponds to cubic and quartic terms in the Hamiltonian and the Fragment
    indices_cq = [i for i, op in enumerate(sorted_ops) if len(op) == 3 or len(op) == 4]

    cost_fn = []

    for i in indices_cq:
        value = H_vector[i] - H_f_vector[i]
        value = np.float64(value)
        value_sq = value**2
        cost_fn.append(value_sq)

    cost = np.real(np.sqrt((sum(cost_fn))))

    return cost


def printx(x):
    global iteration_count
    iteration_count += 1
    cost_value = cost(x)
    with open(f"{name}.txt", "a") as f:
        f.write(f"Step {iteration_count}: cost = {cost_value}\n")
    print(f"Step {iteration_count}: cost = {cost_value}")


current_norm = norm_m(H)
norms.append(current_norm)
print(" Norm of cubic and quartic terms")
print(f"\nInitial Norm : {current_norm}")


numtag = MAX_NUM_FRAGMENTS
tol = 1e-5
num_fragments = len(params)

options = {"maxiter": 2000, "disp": True} 


 Norm of cubic and quartic terms

Initial Norm : 161597.77026327705


## Run Fragmentation

In [8]:
Xs = []

In [9]:
if RUN_FRAGMENTATION:
  start_time = time.time()

  iteration_count = 0

  for k in range(numtag):

      # end decomposition if remaining norm is less than tol
      if current_norm < tol:
          print("Completed!!")
          break

      # obtain fragment

      P, Q, G, L = initial(n)
      X = 1e-3 * flatten_matrices(P, Q, G, L, n)
      Xs.append(X)

      result = minimize(
          cost, X, method="BFGS", tol=1e-7, options=options
      )  # , callback=printx)

      # obtain fragment from parameters in bosonic form
      H_f = fragments(result.x, n)

      # remove fragment from H
      H = subtract(H, H_f)

      current_norm = norm_m(H)

      # save current parameters and current norm
      params.append(result.x)

      norms.append(current_norm)

      num_fragments += 1

      print(
          f"fragment count : {num_fragments}, current norm : {np.round(current_norm, 7)}"
      )


  U, V, L, G, K = b_transform(H, n)  # params for the last fragment

  # Recreate the last fragment (qudratic)
  Hquad = recreate_quadH(U, V, L, G, K, n)

  end_time = time.time()
  elapsed_time = end_time - start_time
  print(f"Runtime : {elapsed_time}")


/home/marlo_2lbtn5r/anaconda3/envs/BSP-WSL2-V3/lib/python3.12/site-packages/scipy/optimize/_minimize.py:779: OptimizeWarning: Desired error not necessarily achieved due to precision loss.
  res = _minimize_bfgs(fun, x0, args, jac, callback, **options)


         Current function value: 11.061671
         Iterations: 517
         Function evaluations: 23840
         Gradient evaluations: 644
fragment count : 1, current norm : 122.3605567


/home/marlo_2lbtn5r/anaconda3/envs/BSP-WSL2-V3/lib/python3.12/site-packages/scipy/optimize/_minimize.py:779: OptimizeWarning: Desired error not necessarily achieved due to precision loss.
  res = _minimize_bfgs(fun, x0, args, jac, callback, **options)


         Current function value: 0.230780
         Iterations: 408
         Function evaluations: 19918
         Gradient evaluations: 538
fragment count : 2, current norm : 0.0532594


/home/marlo_2lbtn5r/anaconda3/envs/BSP-WSL2-V3/lib/python3.12/site-packages/scipy/optimize/_minimize.py:779: OptimizeWarning: Desired error not necessarily achieved due to precision loss.
  res = _minimize_bfgs(fun, x0, args, jac, callback, **options)


         Current function value: 0.139214
         Iterations: 160
         Function evaluations: 8411
         Gradient evaluations: 227
fragment count : 3, current norm : 0.0193804
Runtime : 7630.546504497528


## Post-Processing

### Save to File

In [10]:
import pickle

In [11]:
if RUN_FRAGMENTATION:
  with open(fragment_path, "wb") as f:
      pickle.dump(params, f)
  with open(hquad_path, "wb") as f:
      pickle.dump(Hquad, f)

## import fragments, construct H_eff

In [12]:
import pickle

In [13]:
# def sub_frags(H_target, H_frags):
#   for i in range(len(H_frags)):
#     H_target = subtract(H_target, H_frags[i])
#   return H_target

In [59]:
# from third_party.vibrational_dynamics.util_hamil import extract_coeffs_and_ops
# from openfermion.ops import BosonOperator

# # reconstruct bosonic operator from coefficients and operators
# def reconstruct_boson_operator(coeffs, ops):
#     boson_operator = BosonOperator()
#     for coeff, op in zip(coeffs, ops):
#         boson_operator += BosonOperator(op, coeff)
#     return boson_operator


# def H_quadatic_op(H):
    
#     Hv,ops = extract_coeffs_and_ops(H)    

#     indices_cq = [i for i, op in enumerate(ops) if len(op) < 3]

#     Hvs = list()
#     opss = list()

#     for i in indices_cq:
        
#         Hvs.append(Hv[i])
#         opss.append(ops[i])
        
#     H_qd = reconstruct_boson_operator(Hvs, opss)
    
#     return H_qd

# # Obtain fragments from previously calculated parameters in non-rotated coordinates
# def non_rotated_fragments(name,H,n):
#     # Read the JSON file
#     file_path = f"params_{name}.json"
#     with open(file_path, 'r') as file:
#         data = json.load(file)

#     # Convert the data into a list of NumPy arrays
#     params = [np.array(entry) for entry in data]

#     norms = list()

#     norms.append(norm_m(H))  # norm of the initial H (cubic and quartic)
    
#     # List to store each H2
#     H_frags = []

#     for i in range(len(params)):
#         H2 = fragments(params[i],n)
#         H_frags.append(H2)  # Save each H2 to the list
#         H = subtract(H, H2)
#         norms.append(norm_m(H))

#     H_frags.insert(0,H_quadatic_op(H))
#     #Edited by SVM 09/10/24. Original appended the upto quadratic term.
#     #Now it is inserted in the 1st position.
    
#     return H_frags,norms


In [76]:
# code edited from non_rotated_fragments
MOLECULE = 'H2S'
name = MOLECULE
n = 3

fragment_path = Path.cwd().parent / f'data/{MOLECULE}_Params.pkl'  
H = hamil(name)

with open(fragment_path, 'rb') as file:
    data = pickle.load(file)

# Convert the data into a list of NumPy arrays
params = [np.array(entry) for entry in data]

norms = list()

norms.append(norm_m(H))  # norm of the initial H (cubic and quartic)

# List to store each H2
H_frags = []

for i in range(len(params)):
    H2 = fragments(params[i],n)
    H_frags.append(H2)  # Save each H2 to the list
    H = subtract(H, H2)
    norms.append(norm_m(H))

H_frags.insert(0,H_quadatic_op(H))
H_eff = sum(H_frags)

# H_frags,norms = non_rotated_fragments('CO2',hamil('CO2'),n)
# H_eff = sum(H_frags)
# norms

In [29]:
with open(fragment_path, "rb") as f:
  loaded_params = pickle.load(f)

with open(hquad_path, "rb") as f:
  Hquad = pickle.load(f)

H_frags = [fragments(param, n) for param in loaded_params]
H_0 = Hquad  # H_quadatic_op(H)  #sub_frags(H, H_frags)
H_eff = sum(H_frags) + H_0
H_frags.append(H_0)

In [77]:
len(loaded_params)

3

In [78]:
H_eff

3391.3439033623054 [] +
5.855245336059037 [0] +
-23.046878429147903 [0 0] +
-3.474239873540193 [0 0 0] +
-0.5587113567837183 [0 0 0 0] +
-1.4376346050828417 [0 0 0 1] +
-1.436067854799042 [0 0 0 1^] +
3.767813678802897e-05 [0 0 0 2] +
3.831264115676981e-05 [0 0 0 2^] +
-30.148314706842335 [0 0 1] +
-8.856205916690778 [0 0 1 1] +
-1.0388606947269954e-05 [0 0 1 2] +
-9.537345563768085e-06 [0 0 1 2^] +
-30.147252824450394 [0 0 1^] +
-17.723061678271286 [0 0 1^ 1] +
-8.855431715914007 [0 0 1^ 1^] +
-9.725316047827895e-06 [0 0 1^ 2] +
-1.0218167536526181e-05 [0 0 1^ 2^] +
6.954746938183263e-06 [0 0 2] +
-10.826075692625933 [0 0 2 2] +
7.421908201201915e-06 [0 0 2^] +
-21.65769301031742 [0 0 2^ 2] +
-10.826075690211924 [0 0 2^ 2^] +
-2.654537651079692 [0 1] +
1.0153979714781365 [0 1 1] +
0.546146411367731 [0 1 1 1] +
-6.439749490142526e-06 [0 1 1 2] +
-8.523747276925546e-06 [0 1 1 2^] +
-2.4738779749979055e-05 [0 1 2] +
0.01583041881732962 [0 1 2 2] +
-2.000012590433714e-05 [0 1 2^] +
0.0316

## Bosonic Op Conversion

In [79]:
trunc = TRUNCATION

In [80]:
from openfermion.linalg import boson_operator_sparse

H_frags_sparse = [boson_operator_sparse(H_frag, trunc=trunc) for H_frag in H_frags]

In [81]:
import tensorflow as tf

H_frags_tf = [tf.constant(H_frag.todense(), dtype=tf.complex128) for H_frag in H_frags_sparse]

## Diagonalization

### H_eff

In [82]:
H_eff_sparse = boson_operator_sparse(H_eff, trunc=trunc)

In [83]:
H_eff_eigenenergies = np.linalg.eigvalsh(H_eff_sparse.todense())

In [84]:
H_eff_eigenenergies[0:num_states]

array([3301.74937989, 4563.40119549, 5807.29621186, 5840.04938828,
       5900.04120558])

### H from file

In [85]:
H_original_sparse = boson_operator_sparse(hamil(name), trunc=4)
H_original_eigenenergies = np.linalg.eigvalsh(H_original_sparse.todense())
H_original_eigenenergies[:num_states]

array([3301.73424272, 4563.38579267, 5807.26448734, 5840.03581722,
       5899.92397929])

# Run QumVQD

In [25]:
from helpers.QumVQD_Helpers import *

In [26]:
beta_factor = 4.0 * norm_m(H_eff)
H_eff_tf = [tf.constant(H_eff_sparse.todense(), dtype=tf.complex128)]
H_ops = H_eff_tf # Can use H_frags_tf or H_eff_tf
coeffs = np.ones(len(H_ops))

energies, state_params, results, psis = find_excited_states_multi_qumode(
    H_ops = H_ops, coeffs = coeffs, ndepth = ndepth,
    num_states = num_states, beta_factor=beta_factor,
    niter=niter, threshold=threshold, nfocks=nfocks, save_files=True, save_dir=f"output_data/{name}_VQD_results"
)

print("Energies found:", energies)

num_prev_states = 0

Finding state 0...
  [saved checkpoint to output_data/CO2_VQD_results after state 0]
State 0 energy: 4455.038184
Converged: False

Finding state 1...
Overlap with state 0: 2.087746e-03
  [saved checkpoint to output_data/CO2_VQD_results after state 1]
State 1 energy: 10220.956894
Converged: False

Finding state 2...
Overlap with state 0: 4.831202e-05
Overlap with state 1: 6.054694e-04
  [saved checkpoint to output_data/CO2_VQD_results after state 2]
State 2 energy: 10643.216894
Converged: False

Finding state 3...
Overlap with state 0: 3.307227e-03
Overlap with state 1: 2.304475e-03
Overlap with state 2: 1.467632e-03
  [saved checkpoint to output_data/CO2_VQD_results after state 3]
State 3 energy: 9934.535735
Converged: False

Finding state 4...
Overlap with state 0: 1.595405e-03
Overlap with state 1: 5.331560e-03
Overlap with state 2: 1.828630e-03
Overlap with state 3: 6.562929e-04
  [saved checkpoint to output_data/CO2_VQD_results after state 4]
State 4 energy: 10

In [27]:
en = [float(energy) for energy in energies]
print(en)

[4455.038183680085, 10220.956893837623, 10643.216893985838, 9934.535734917905, 10166.240109009388]
